In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [3]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [4]:
CCBal = pd.read_csv('data/Internal/credit_card_balance.csv')

In [5]:
CCBal = CCBal.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})
CCBal.dtypes

SK_ID_PREV                    string[python]
SK_ID_CURR                    string[python]
MONTHS_BALANCE                         int64
AMT_BALANCE                          float64
AMT_CREDIT_LIMIT_ACTUAL                int64
AMT_DRAWINGS_ATM_CURRENT             float64
AMT_DRAWINGS_CURRENT                 float64
AMT_DRAWINGS_OTHER_CURRENT           float64
AMT_DRAWINGS_POS_CURRENT             float64
AMT_INST_MIN_REGULARITY              float64
AMT_PAYMENT_CURRENT                  float64
AMT_PAYMENT_TOTAL_CURRENT            float64
AMT_RECEIVABLE_PRINCIPAL             float64
AMT_RECIVABLE                        float64
AMT_TOTAL_RECEIVABLE                 float64
CNT_DRAWINGS_ATM_CURRENT             float64
CNT_DRAWINGS_CURRENT                   int64
CNT_DRAWINGS_OTHER_CURRENT           float64
CNT_DRAWINGS_POS_CURRENT             float64
CNT_INSTALMENT_MATURE_CUM            float64
NAME_CONTRACT_STATUS                  object
SK_DPD                                 int64
SK_DPD_DEF

In [6]:
CCBal.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'], ascending=[True, True, True], inplace=True)
CCBal = CCBal.reset_index(drop=True)

In [ ]:
#temp = CCBal.groupby('SK_ID_PREV')[['MONTHS_BALANCE']].agg(['min', 'max']).reset_index()

In [13]:
CCBal = CCBal.merge(base, on='SK_ID_CURR', how='left')

In [14]:
CCBal = CCBal[CCBal['TARGET'].notna()]

In [15]:
temp = CCBal[['SK_ID_PREV','SK_ID_CURR']].drop_duplicates().groupby('SK_ID_CURR').size().reset_index(name='count')

In [10]:
temp = temp[temp['count'] > 1]

In [16]:
def create_yoy_features(df):
    """
    Engineers Year-over-Year (YoY) change features for a credit dataset.

    This function calculates the YoY percentage change for financial amounts and
    the YoY absolute difference for metrics like 'days past due'.

    Args:
        df (pd.DataFrame): The input DataFrame containing credit history.
                           It must contain 'SK_ID_CURR' for client ID,
                           'MONTHS_BALANCE' for the timeline, and the metrics
                           to be transformed.

    Returns:
        pd.DataFrame: The DataFrame with new YoY feature columns added.
    """
    # --- Pre-computation & Sorting ---
    # For this calculation to work correctly, the data must be sorted by the
    # individual credit account (SK_ID_CURR) and the timeline (MONTHS_BALANCE).

    # --- Feature Engineering ---
    # We group by each client to ensure the YoY calculation is done within
    # the same credit account.
    grouped = df.groupby(['SK_ID_PREV','SK_ID_CURR'])

    # List of columns for which we want to calculate YoY percentage change.
    # These are typically monetary values where relative change is important.
    amount_cols = [
        'AMT_BALANCE',
        'AMT_CREDIT_LIMIT_ACTUAL',
        'AMT_DRAWINGS_CURRENT',
        'AMT_PAYMENT_TOTAL_CURRENT'
    ]
    df[amount_cols].clip(lower=0, inplace=True) 
    # List of columns for which we want to calculate YoY absolute difference.
    # These are often counts or statuses where the raw change is more informative.
    other_cols = [
        'SK_DPD',
        'CNT_DRAWINGS_CURRENT',
        'CNT_INSTALMENT_MATURE_CUM'
    ]

    # --- Calculate YoY Percentage Change for Amounts ---
    for col in amount_cols:
        # Get the value from 12 months ago. The shift(12) function moves the
        # data down by 12 rows within each group.
        prev_year_val = grouped[col].shift(12)

        # Calculate the YoY percentage change.
        # (current - past) / past
        # We use np.where to avoid division by zero if the past value was 0.
        # If the denominator is 0, we'll record a large change (1) if the current
        # value is positive, and 0 otherwise.
        df[f'{col}_YOY_PCT_CHANGE'] = (df[col] - prev_year_val) / np.where(prev_year_val == 0, 1, prev_year_val)
        # Replace infinite values (from division by a very small number) with NaN
        # df[f'{col}_YOY_PCT_CHANGE'].clip(lower=-1, upper=1, inplace=True)
        df[f'{col}_YOY_PCT_CHANGE'].replace([np.inf, -np.inf], np.nan, inplace=True)


    # --- Calculate YoY Absolute Difference for Other Metrics ---
    for col in other_cols:
        # Get the value from 12 months ago.
        prev_year_val = grouped[col].shift(12)

        # Calculate the absolute difference: current_value - past_value
        df[f'{col}_YOY_ABS_CHANGE'] = df[col] - prev_year_val

    # --- Final Touches ---
    # The shift operation creates NaNs for the first 12 months of data for each
    # account. For modeling, it's often best to fill these with 0, assuming
    # no change when historical data is not available.
    yoy_cols = [col for col in df.columns if '_YOY_' in col]
    df[yoy_cols] = df[yoy_cols].fillna(0)

    print("YoY features created successfully. New columns:")
    print(yoy_cols)

    return df


Impute data

In [17]:
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_ATM_CURRENT'].isna()), 'AMT_DRAWINGS_ATM_CURRENT'] = 0
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_OTHER_CURRENT'].isna()), 'AMT_DRAWINGS_OTHER_CURRENT'] = 0
CCBal.loc[(CCBal['AMT_DRAWINGS_CURRENT'] == 0) & (CCBal['AMT_DRAWINGS_POS_CURRENT'].isna()), 'AMT_DRAWINGS_POS_CURRENT'] = 0

In [18]:
CCBal[CCBal['AMT_DRAWINGS_CURRENT'] > 0][CCBal['AMT_DRAWINGS_ATM_CURRENT'] > 0] # AMT_DRAWINGS_OTHER_CURRENT AMT_DRAWINGS_POS_CURRENT

C:\Users\domjo\AppData\Local\Temp\ipykernel_18264\4122397814.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  CCBal[CCBal['AMT_DRAWINGS_CURRENT'] > 0][CCBal['AMT_DRAWINGS_ATM_CURRENT'] > 0] # AMT_DRAWINGS_OTHER_CURRENT AMT_DRAWINGS_POS_CURRENT


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF,TARGET
6,1843384,100011,-75,189000.000,180000,180000.0,180000.000,0.0,0.000,NaN,...,189000.000,4.0,4,0.0,0.0,NaN,Active,0,0,0.0
354,1557583,100043,-25,435861.585,427500,67500.0,67500.000,0.0,0.000,18361.17,...,433170.585,2.0,2,0.0,0.0,7.0,Active,0,0,0.0
444,1060558,100048,-2,27302.400,225000,27000.0,27000.000,0.0,0.000,0.00,...,25956.900,1.0,1,0.0,0.0,17.0,Active,0,0,0.0
452,1209367,100049,-8,10397.115,90000,9000.0,9000.000,0.0,0.000,0.00,...,9051.615,1.0,1,0.0,0.0,0.0,Active,0,0,1.0
453,1209367,100049,-7,82449.135,90000,72000.0,73990.935,0.0,1990.935,0.00,...,78412.635,3.0,6,0.0,3.0,0.0,Active,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3840185,1595171,456247,-93,186025.230,180000,96750.0,96750.000,0.0,0.000,4500.00,...,186025.230,4.0,4,0.0,0.0,6.0,Active,0,0,0.0
3840190,1595171,456247,-88,185570.505,180000,9000.0,9000.000,0.0,0.000,9000.00,...,185570.505,2.0,2,0.0,0.0,11.0,Active,0,0,0.0
3840199,1595171,456247,-79,4143.420,180000,6750.0,6750.000,0.0,0.000,0.00,...,4143.420,2.0,2,0.0,0.0,17.0,Active,0,0,0.0
3840202,1595171,456247,-76,96776.640,180000,90000.0,90000.000,0.0,0.000,0.00,...,96776.640,2.0,2,0.0,0.0,19.0,Active,0,0,0.0


In [19]:
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_ATM_CURRENT'].isna()), 'CNT_DRAWINGS_ATM_CURRENT'] = 0
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_POS_CURRENT'].isna()), 'CNT_DRAWINGS_POS_CURRENT'] = 0
CCBal.loc[(CCBal['CNT_DRAWINGS_CURRENT'] == 0) & (CCBal['CNT_DRAWINGS_OTHER_CURRENT'].isna()), 'CNT_DRAWINGS_OTHER_CURRENT'] = 0

In [21]:
CCBal['AMT_PAYMENT_CURRENT'] = CCBal.groupby('SK_ID_PREV')['AMT_PAYMENT_CURRENT'].transform(
    lambda x: x.ffill(limit=3).bfill(limit=3)
) #.reset_index()

YOY data

In [22]:
CCBal = create_yoy_features(CCBal)

C:\Users\domjo\AppData\Local\Temp\ipykernel_18264\3737033918.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[amount_cols].clip(lower=0, inplace=True)
C:\Users\domjo\AppData\Local\Temp\ipykernel_18264\3737033918.py:57: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[f'{col}_YOY_PCT_CHANGE'].replace([np.inf, -np.inf], np.nan, inplace=Tr

YoY features created successfully. New columns:
['AMT_BALANCE_YOY_PCT_CHANGE', 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE', 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE', 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE', 'SK_DPD_YOY_ABS_CHANGE', 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE', 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE']


Create Ratios

In [23]:
CCBal['UTILIZATION_RATIO'] = np.where(CCBal['AMT_CREDIT_LIMIT_ACTUAL'] >= 0, 
                                      ( CCBal['AMT_BALANCE'] / (1 + CCBal['AMT_CREDIT_LIMIT_ACTUAL']) ) * 100, 
                                      np.nan)
CCBal['PAYMENT_RATIO'] = np.where((CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + CCBal['AMT_TOTAL_RECEIVABLE']) >= 0 , 
                                  ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] / (1 + CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + CCBal['AMT_TOTAL_RECEIVABLE']) ) * 100 ,
                                  np.nan)

CCBal['DRAWING_TO_PYMNT_RATIO'] = np.where(CCBal['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                           ( CCBal['AMT_DRAWINGS_CURRENT'] / ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                           np.nan)

CCBal['INT_COVERAGE_RATIO']    = np.where(CCBal['AMT_PAYMENT_TOTAL_CURRENT'] >= 0 , 
                                          ((CCBal['AMT_TOTAL_RECEIVABLE']-CCBal['AMT_RECEIVABLE_PRINCIPAL']) / ( CCBal['AMT_PAYMENT_TOTAL_CURRENT'] + 1) ) * 100 ,
                                          np.nan)   
CCBal['POS_DRAWING_AMT_RATIO'] = np.where(CCBal['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                         (CCBal['AMT_DRAWINGS_POS_CURRENT'] / (1 + CCBal['AMT_DRAWINGS_CURRENT']) ) * 100, 
                                          np.nan)

CCBal['ATM_DRAWING_AMT_RATIO'] = np.where(CCBal['AMT_DRAWINGS_CURRENT'] >= 0 , 
                                          (CCBal['AMT_DRAWINGS_ATM_CURRENT'] / (1 + CCBal['AMT_DRAWINGS_CURRENT']) ) * 100  , 
                                          np.nan)

CCBal['POS_DRAWING_CNT_RATIO'] = np.where(CCBal['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                          (CCBal['CNT_DRAWINGS_POS_CURRENT'] / (1 + CCBal['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                          np.nan)
  
CCBal['ATM_DRAWING_CNT_RATIO'] = np.where(CCBal['CNT_DRAWINGS_CURRENT'] >= 0 , 
                                          ( CCBal['CNT_DRAWINGS_ATM_CURRENT'] / (1 + CCBal['CNT_DRAWINGS_CURRENT']) ) * 100, 
                                            np.nan)  
CCBal['ISNULL_DRAWINGS_CURRENT_FLAG'] = CCBal['AMT_DRAWINGS_CURRENT'].isnull() * 1
CCBal['DPD_GE_1'] = ( CCBal['SK_DPD'] >= 1 ) * 1
CCBal['DPD_GE_7'] = ( CCBal['SK_DPD'] >= 7 ) * 1
CCBal['DPD_GE_30'] = ( CCBal['SK_DPD'] >= 30 ) * 1
CCBal['DPD_UTP'] = ( (CCBal['SK_DPD'] >= 1) & (CCBal['AMT_BALANCE'] >= 100) ) * 1


In [16]:
# drop columns with very high NaN values
#CCBal.drop(columns=['POS_DRAWING_AMT_RATIO','ATM_DRAWING_AMT_RATIO', 'POS_DRAWING_CNT_RATIO', 'ATM_DRAWING_CNT_RATIO'], inplace=True)

In [24]:
[i for i in CCBal.columns]

['SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'AMT_BALANCE',
 'AMT_CREDIT_LIMIT_ACTUAL',
 'AMT_DRAWINGS_ATM_CURRENT',
 'AMT_DRAWINGS_CURRENT',
 'AMT_DRAWINGS_OTHER_CURRENT',
 'AMT_DRAWINGS_POS_CURRENT',
 'AMT_INST_MIN_REGULARITY',
 'AMT_PAYMENT_CURRENT',
 'AMT_PAYMENT_TOTAL_CURRENT',
 'AMT_RECEIVABLE_PRINCIPAL',
 'AMT_RECIVABLE',
 'AMT_TOTAL_RECEIVABLE',
 'CNT_DRAWINGS_ATM_CURRENT',
 'CNT_DRAWINGS_CURRENT',
 'CNT_DRAWINGS_OTHER_CURRENT',
 'CNT_DRAWINGS_POS_CURRENT',
 'CNT_INSTALMENT_MATURE_CUM',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF',
 'TARGET',
 'AMT_BALANCE_YOY_PCT_CHANGE',
 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
 'SK_DPD_YOY_ABS_CHANGE',
 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE',
 'UTILIZATION_RATIO',
 'PAYMENT_RATIO',
 'DRAWING_TO_PYMNT_RATIO',
 'INT_COVERAGE_RATIO',
 'POS_DRAWING_AMT_RATIO',
 'ATM_DRAWING_AMT_RATIO',
 'POS_DRAWING_CNT_RAT

last 3 Year aggregates

In [25]:
CCBal_Aggregate = CCBal[CCBal['MONTHS_BALANCE']>=-36].groupby(['SK_ID_PREV', 'SK_ID_CURR']).agg(
    CREDITCARD_MAX_DURATION=('MONTHS_BALANCE','min'),
 #   CREDITCARD_COUNT=('SK_ID_PREV','nunique'),
    CREDITCARD_AVG_LIMIT_AMOUNT=('AMT_CREDIT_LIMIT_ACTUAL','mean'),
    CREDITCARD_AVG_BALANCE_AMOUNT=('AMT_BALANCE','mean'),
    CREDITCARD_AVG_DRAWING_AMOUNT=('AMT_DRAWINGS_CURRENT','mean'),
    CREDITCARD_AVG_PAYMENT_AMOUNT=('AMT_PAYMENT_TOTAL_CURRENT','mean'),
    CREDITCARD_MAX_UTILIZATION_RATE=('UTILIZATION_RATIO','mean'),
    CREDITCARD_AVG_PAYMENT_RATE=('PAYMENT_RATIO','mean'),
    CREDITCARD_AVG_DRAWING_TO_PAYMENT_RATE=('DRAWING_TO_PYMNT_RATIO','mean'),
    CREDITCARD_AVG_INTCOVERAGE_RATE=('INT_COVERAGE_RATIO','mean'),
    CREDITCARD_AVG_POSAMT_RATE=('POS_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_ATMAMT_RATE=('ATM_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_POSCOUNT_RATE=('POS_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_ATMCOUNT_RATE=('ATM_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_REPAYMENTCOUNT_RATE=('CNT_INSTALMENT_MATURE_CUM', lambda x: x[x > 0].count()/x.count()),
    CREDITCARD_ISNULL_DRAWINGS_CURRENT_FLAG=('ISNULL_DRAWINGS_CURRENT_FLAG','max'),
    CREDITCARD_MAX_DPD=('DPD_GE_7','max'),
    CREDITCARD_MAX_DPDUTP=('DPD_UTP','max'),
    CREDITCARD_DAYS_IN_DPD_RATIO=('SK_DPD', lambda x: x[x >= 1].count()/x.count() ),
    # Year-over-Year (YoY) Change Features
    CREDITCARD_AVG_AMT_BALANCE_YOY_PCT_CHANGE = ('AMT_BALANCE_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE = ('AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE = ('AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE = ('AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE = ('CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE = ('CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_SK_DPD_YOY_ABS_CHANGE = ('SK_DPD_YOY_ABS_CHANGE', 'mean')
).reset_index()

In [26]:
[i for i in CCBal.columns]

['SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'AMT_BALANCE',
 'AMT_CREDIT_LIMIT_ACTUAL',
 'AMT_DRAWINGS_ATM_CURRENT',
 'AMT_DRAWINGS_CURRENT',
 'AMT_DRAWINGS_OTHER_CURRENT',
 'AMT_DRAWINGS_POS_CURRENT',
 'AMT_INST_MIN_REGULARITY',
 'AMT_PAYMENT_CURRENT',
 'AMT_PAYMENT_TOTAL_CURRENT',
 'AMT_RECEIVABLE_PRINCIPAL',
 'AMT_RECIVABLE',
 'AMT_TOTAL_RECEIVABLE',
 'CNT_DRAWINGS_ATM_CURRENT',
 'CNT_DRAWINGS_CURRENT',
 'CNT_DRAWINGS_OTHER_CURRENT',
 'CNT_DRAWINGS_POS_CURRENT',
 'CNT_INSTALMENT_MATURE_CUM',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF',
 'TARGET',
 'AMT_BALANCE_YOY_PCT_CHANGE',
 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
 'SK_DPD_YOY_ABS_CHANGE',
 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE',
 'UTILIZATION_RATIO',
 'PAYMENT_RATIO',
 'DRAWING_TO_PYMNT_RATIO',
 'INT_COVERAGE_RATIO',
 'POS_DRAWING_AMT_RATIO',
 'ATM_DRAWING_AMT_RATIO',
 'POS_DRAWING_CNT_RAT

Last Snapshot

In [27]:
lastsnapshotcols = [
 'SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'AMT_BALANCE',
 'AMT_CREDIT_LIMIT_ACTUAL',
 'AMT_DRAWINGS_ATM_CURRENT',
 'AMT_DRAWINGS_CURRENT',
#'AMT_DRAWINGS_OTHER_CURRENT',
 'AMT_DRAWINGS_POS_CURRENT',
#'AMT_INST_MIN_REGULARITY',
 'AMT_PAYMENT_CURRENT',
 'AMT_PAYMENT_TOTAL_CURRENT',
 'AMT_RECEIVABLE_PRINCIPAL',
 'AMT_RECIVABLE',
 'AMT_TOTAL_RECEIVABLE',
 'CNT_DRAWINGS_ATM_CURRENT',
 'CNT_DRAWINGS_CURRENT',
#'CNT_DRAWINGS_OTHER_CURRENT',
 'CNT_DRAWINGS_POS_CURRENT',
 'CNT_INSTALMENT_MATURE_CUM',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF',
 'AMT_BALANCE_YOY_PCT_CHANGE',
 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
 'SK_DPD_YOY_ABS_CHANGE',
 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE',
 'UTILIZATION_RATIO',
 'PAYMENT_RATIO',
 'DRAWING_TO_PYMNT_RATIO',
 'INT_COVERAGE_RATIO',
 'POS_DRAWING_AMT_RATIO',
 'ATM_DRAWING_AMT_RATIO',
 'POS_DRAWING_CNT_RATIO',
 'ATM_DRAWING_CNT_RATIO',
 'ISNULL_DRAWINGS_CURRENT_FLAG',
 'DPD_GE_1',
 'DPD_GE_7',
 'DPD_GE_30',
 'DPD_UTP'
]

In [28]:
CCBal_Last_Snapshot= CCBal.groupby(['SK_ID_PREV', 'SK_ID_CURR']).last().reset_index()[lastsnapshotcols]
CCBal_Last_Snapshot.columns = ['CREDITCARD_LAST_MONTH_' + col if col not in ['SK_ID_CURR', 'SK_ID_PREV'] else col for col in CCBal_Last_Snapshot.columns]

First Snapshot

In [29]:
firstsnapshotcols = [
 'SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'AMT_BALANCE',
 'AMT_CREDIT_LIMIT_ACTUAL',
 'AMT_DRAWINGS_ATM_CURRENT',
 'AMT_DRAWINGS_CURRENT',
#'AMT_DRAWINGS_OTHER_CURRENT',
 'AMT_DRAWINGS_POS_CURRENT',
#'AMT_INST_MIN_REGULARITY',
 'AMT_PAYMENT_CURRENT',
 'AMT_PAYMENT_TOTAL_CURRENT',
 'AMT_RECEIVABLE_PRINCIPAL',
 'AMT_RECIVABLE',
 'AMT_TOTAL_RECEIVABLE',
 'CNT_DRAWINGS_ATM_CURRENT',
 'CNT_DRAWINGS_CURRENT',
#'CNT_DRAWINGS_OTHER_CURRENT',
 'CNT_DRAWINGS_POS_CURRENT',
 'CNT_INSTALMENT_MATURE_CUM',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF',
 'AMT_BALANCE_YOY_PCT_CHANGE',
 'AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
 'AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
 'AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
 'SK_DPD_YOY_ABS_CHANGE',
 'CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
 'CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE',
 'UTILIZATION_RATIO',
 'PAYMENT_RATIO',
 'DRAWING_TO_PYMNT_RATIO',
 'INT_COVERAGE_RATIO',
 'POS_DRAWING_AMT_RATIO',
 'ATM_DRAWING_AMT_RATIO',
 'POS_DRAWING_CNT_RATIO',
 'ATM_DRAWING_CNT_RATIO',
 'ISNULL_DRAWINGS_CURRENT_FLAG',
 'DPD_GE_1',
 'DPD_GE_7',
 'DPD_GE_30',
 'DPD_UTP'
]

In [30]:
CCBal_First_Snapshot= CCBal.groupby(['SK_ID_PREV', 'SK_ID_CURR']).first().reset_index()[firstsnapshotcols]
CCBal_First_Snapshot.columns = ['CREDITCARD_FIRST_MONTH_' + col if col not in ['SK_ID_CURR', 'SK_ID_PREV'] else col for col in CCBal_First_Snapshot.columns]

Create days to DPD

In [ ]:
# create days to DPD > 7
CCBal_DPDge7 = CCBal[CCBal['SK_DPD'] >= 7].groupby(['SK_ID_PREV', 'SK_ID_CURR']).first().reset_index()[['SK_ID_PREV', 'SK_ID_CURR','MONTHS_BALANCE']] #.rename(columns={'MONTHS_BALANCE': 'CREDITCARD_DAYS_TO_DPD_GE7'}) 

In [32]:
CCBal_DPDge7 = CCBal_DPDge7.merge(CCBal_First_Snapshot[['SK_ID_PREV', 'SK_ID_CURR','CREDITCARD_FIRST_MONTH_MONTHS_BALANCE']], 
                                  on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
CCBal_DPDge7['CREDITCARD_DAYS_TO_DPD_GE7'] = CCBal_DPDge7['MONTHS_BALANCE'] - CCBal_DPDge7['CREDITCARD_FIRST_MONTH_MONTHS_BALANCE']

In [33]:
# create days to DPD >= 1
CCBal_DPDge1 = CCBal[CCBal['SK_DPD'] >= 1].groupby(['SK_ID_PREV', 'SK_ID_CURR']).first().reset_index()[['SK_ID_PREV', 'SK_ID_CURR','MONTHS_BALANCE']] #.rename(columns={'MONTHS_BALANCE': 'CREDITCARD_DAYS_TO_DPD_GE1'})

In [34]:
CCBal_DPDge1 = CCBal_DPDge1.merge(CCBal_First_Snapshot[['SK_ID_PREV', 'SK_ID_CURR','CREDITCARD_FIRST_MONTH_MONTHS_BALANCE']], 
                                  on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
CCBal_DPDge1['CREDITCARD_DAYS_TO_DPD_GE1'] = CCBal_DPDge1['MONTHS_BALANCE'] - CCBal_DPDge1['CREDITCARD_FIRST_MONTH_MONTHS_BALANCE']

Last One Year Data

In [35]:
# last year data
CCBal_LastYear = CCBal[CCBal['MONTHS_BALANCE'] >= -12].reset_index()

In [41]:
CCBal_LastYear_Aggregate = CCBal_LastYear.groupby(['SK_ID_PREV', 'SK_ID_CURR']).agg(
    CREDITCARD_AVG_LY_LIMIT_AMOUNT=('AMT_CREDIT_LIMIT_ACTUAL','mean'),
    CREDITCARD_AVG_LY_BALANCE_AMOUNT=('AMT_BALANCE','mean'),
    CREDITCARD_AVG_LY_DRAWING_AMOUNT=('AMT_DRAWINGS_CURRENT','mean'),
    CREDITCARD_AVG_LY_PAYMENT_AMOUNT=('AMT_PAYMENT_TOTAL_CURRENT','mean'),
    CREDITCARD_MAX_LY_UTILIZATION_RATE=('UTILIZATION_RATIO','mean'),
    CREDITCARD_AVG_LY_PAYMENT_RATE=('PAYMENT_RATIO','mean'),
    CREDITCARD_AVG_LY_DRAWING_TO_PAYMENT_RATE=('DRAWING_TO_PYMNT_RATIO','mean'),
    CREDITCARD_AVG_LY_INTCOVERAGE_RATE=('INT_COVERAGE_RATIO','mean'),
    CREDITCARD_AVG_LY_POSAMT_RATE=('POS_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_LY_ATMAMT_RATE=('ATM_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_LY_POSCOUNT_RATE=('POS_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_LY_ATMCOUNT_RATE=('ATM_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_LY_REPAYMENTCOUNT_RATE=('CNT_INSTALMENT_MATURE_CUM', lambda x: x[x > 0].count()/x.count()),
    CREDITCARD_MAX_LY_DPD=('DPD_GE_7','max'),
    CREDITCARD_MAX_LY_DPDUTP=('DPD_UTP','max'),
    CREDITCARD_LY_DAYS_IN_DPD_RATIO=('SK_DPD', lambda x: x[x >= 1].count()/x.count() ) ,

    CREDITCARD_AVG_LY_AMT_BALANCE_YOY_PCT_CHANGE = ('AMT_BALANCE_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE = ('AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE = ('AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE = ('AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE = ('CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE = ('CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_LY_SK_DPD_YOY_ABS_CHANGE = ('SK_DPD_YOY_ABS_CHANGE', 'mean')
)

In [37]:
# last semester data
CCBal_LastSem = CCBal[CCBal['MONTHS_BALANCE'] >= -6].reset_index()

In [40]:
CCBal_LastSem_Aggregate = CCBal_LastSem.groupby(['SK_ID_PREV', 'SK_ID_CURR']).agg(
    CREDITCARD_AVG_LS_LIMIT_AMOUNT=('AMT_CREDIT_LIMIT_ACTUAL','mean'),
    CREDITCARD_AVG_LS_BALANCE_AMOUNT=('AMT_BALANCE','mean'),
    CREDITCARD_AVG_LS_DRAWING_AMOUNT=('AMT_DRAWINGS_CURRENT','mean'),
    CREDITCARD_AVG_LS_PAYMENT_AMOUNT=('AMT_PAYMENT_TOTAL_CURRENT','mean'),
    CREDITCARD_MAX_LS_UTILIZATION_RATE=('UTILIZATION_RATIO','mean'),
    CREDITCARD_AVG_LS_PAYMENT_RATE=('PAYMENT_RATIO','mean'),
    CREDITCARD_AVG_LS_DRAWING_TO_PAYMENT_RATE=('DRAWING_TO_PYMNT_RATIO','mean'),
    CREDITCARD_AVG_LS_INTCOVERAGE_RATE=('INT_COVERAGE_RATIO','mean'),
    CREDITCARD_AVG_LS_POSAMT_RATE=('POS_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_LS_ATMAMT_RATE=('ATM_DRAWING_AMT_RATIO','mean'),
    CREDITCARD_AVG_LS_POSCOUNT_RATE=('POS_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_LS_ATMCOUNT_RATE=('ATM_DRAWING_CNT_RATIO','mean'),
    CREDITCARD_AVG_LS_REPAYMENTCOUNT_RATE=('CNT_INSTALMENT_MATURE_CUM', lambda x: x[x > 0].count()/x.count()),
    CREDITCARD_MAX_LS_DPD=('DPD_GE_7','max'),
    CREDITCARD_MAX_LS_DPDUTP=('DPD_UTP','max'),
    CREDITCARD_LS_DAYS_IN_DPD_RATIO=('SK_DPD', lambda x: x[x >= 1].count()/x.count() ),  

    CREDITCARD_AVG_LS_AMT_BALANCE_YOY_PCT_CHANGE = ('AMT_BALANCE_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE = ('AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE = ('AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE = ('AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE = ('CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE = ('CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE', 'mean'),
    CREDITCARD_AVG_LS_SK_DPD_YOY_ABS_CHANGE = ('SK_DPD_YOY_ABS_CHANGE', 'mean')
)

Merge All data

In [43]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_LastYear_Aggregate, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

In [44]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_LastSem_Aggregate, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

In [45]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_DPDge7[['SK_ID_PREV', 'SK_ID_CURR', 'CREDITCARD_DAYS_TO_DPD_GE7']], on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_DPDge1[['SK_ID_PREV', 'SK_ID_CURR', 'CREDITCARD_DAYS_TO_DPD_GE1']], on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

In [47]:
CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE1'] = CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE1'].fillna(CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE1'].max()+1)
CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE7'] = CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE7'].fillna(CCBal_Aggregate['CREDITCARD_DAYS_TO_DPD_GE7'].max()+1)

In [48]:
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_Last_Snapshot, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
CCBal_Aggregate = CCBal_Aggregate.merge(CCBal_First_Snapshot, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

Join Default Indicator

In [50]:
CCBal_Aggregate = CCBal_Aggregate.merge(base, on='SK_ID_CURR', how='left')

In [52]:
CCBal_Aggregate.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\creditcard_processed.parquet")